# Ames Mutagenicity Prediction: QSAR for Drug Safety Screening
### Classification with SMOTE, ROC-AUC, and Structural Alert Analysis

**Author:** Shehan Makani | ChemeNova LLC | NJIT Tech MBA  
**GitHub:** [github.com/shehanmakani/cheminformatics-ml](https://github.com/shehanmakani/cheminformatics-ml)  
**Series:** Notebook 4 — Molecular Property Prediction

---

## What Is the Ames Test?

The Ames test (Salmonella typhimurium reverse mutation assay, OECD 471) is the primary regulatory screen for chemical mutagenicity. It asks: does this compound cause DNA mutations? A positive result is a serious flag — mutagenic compounds are often carcinogenic.

Every drug candidate, food additive, cosmetic ingredient, and industrial chemical entering regulated markets needs mutagenicity data. The test takes days and costs hundreds to thousands of dollars per compound. Computational prediction lets you screen thousands of candidates before committing to wet chemistry.

**What makes this notebook different from the previous three:**
- This is **classification**, not regression — completely different evaluation framework
- The dataset is **imbalanced** (44% positive) — we use SMOTE to address this
- **Recall matters more than accuracy** — a missed mutagen (false negative) is far more dangerous than a false positive in toxicology screening
- **Structural alerts** — we engineer explicit features for known mutagenic substructures (nitro groups, nitroso groups, aromatic amines, alkyl halides)
- **ROC-AUC and Precision-Recall curves** replace RMSE as the performance metric

**Key result:** Logistic Regression achieves AUC=0.987, recall=0.909 — surprisingly, the simplest model wins. This is chemically interpretable: mutagenicity is largely determined by a small number of structural features that linear boundaries capture well.


## 1. Setup


In [ ]:
import subprocess, sys
for pkg in ['rdkit','imbalanced-learn']:
    try:
        __import__(pkg.replace('-','_').replace('imbalanced_learn','imblearn'))
    except ImportError:
        subprocess.run([sys.executable,'-m','pip','install',pkg,'-q'])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, Crippen

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_curve,
                              accuracy_score, f1_score, precision_score, recall_score,
                              confusion_matrix, average_precision_score)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap

TEAL='#00c8b4'; GOLD='#f0a500'; CORAL='#ff6b6b'; VIOLET='#9b59b6'
print('All imports successful.')


## 2. Dataset — 122 Compounds, Known Ames Results

Curated from Hansen et al. (2009) and Kazius et al. (2005). Covers:

**Mutagens (label=1):** nitroaromatics, aromatic amines, PAH metabolites, alkylating agents, N-nitroso compounds, heterocyclic amines, azo dyes

**Non-mutagens (label=0):** aliphatics, simple aromatics, alcohols, drugs (aspirin, ibuprofen, caffeine), natural products, common solvents

Class balance: ~43% mutagenic — mild imbalance, we use SMOTE to correct it on the training set.

**Engineering structural alert features:** Beyond generic descriptors, we add 4 binary features for known mutagenic substructures:
- `HasNitroGroup` — nitro (-NO₂) attached to aromatic ring: 100% mutagenic rate
- `HasNitroso` — nitroso (-N=O): 100% mutagenic rate
- `HasAromaticAmine` — primary amine on aromatic ring: 30% mutagenic rate
- `HasAlkylHalide` — halogen on aliphatic carbon: 43% mutagenic rate


In [ ]:
rows = [
    # Mutagens (1)
    ('2-nitrofluorene','O=[N+]([O-])c1ccc2cc3ccccc3cc2c1',1),
    ('2-acetylaminofluorene','CC(=O)Nc1ccc2cc3ccccc3cc2c1',1),
    ('4-nitroquinoline N-oxide','O=[N+]([O-])c1ccc2ncccc2c1',1),
    ('1-nitropyrene','O=[N+]([O-])c1ccc2ccc3cccc4ccc1c2c34',1),
    ('nitrobenzene','O=[N+]([O-])c1ccccc1',1),
    ('2-nitrotoluene','Cc1ccccc1[N+](=O)[O-]',1),
    ('4-nitrotoluene','Cc1ccc([N+](=O)[O-])cc1',1),
    ('2,4-dinitrotoluene','Cc1ccc([N+](=O)[O-])cc1[N+](=O)[O-]',1),
    ('2,6-dinitrotoluene','Cc1c([N+](=O)[O-])cccc1[N+](=O)[O-]',1),
    ('1,3-dinitrobenzene','O=[N+]([O-])c1cccc([N+](=O)[O-])c1',1),
    ('4-nitroanisole','COc1ccc([N+](=O)[O-])cc1',1),
    ('4-nitrophenol','Oc1ccc([N+](=O)[O-])cc1',1),
    ('4-nitroaniline','Nc1ccc([N+](=O)[O-])cc1',1),
    ('2-nitroaniline','Nc1ccccc1[N+](=O)[O-]',1),
    ('3-nitroaniline','Nc1cccc([N+](=O)[O-])c1',1),
    ('2-naphthylamine','Nc1ccc2ccccc2c1',1),
    ('1-naphthylamine','Nc1cccc2ccccc12',1),
    ('benzidine','Nc1ccc(-c2ccc(N)cc2)cc1',1),
    ('4-aminobiphenyl','Nc1ccc(-c2ccccc2)cc1',1),
    ('2-aminofluorene','Nc1ccc2cc3ccccc3cc2c1',1),
    ('aniline','Nc1ccccc1',1),
    ('o-toluidine','Cc1ccccc1N',1),
    ('m-toluidine','Cc1cccc(N)c1',1),
    ('p-toluidine','Cc1ccc(N)cc1',1),
    ('4-chloroaniline','Nc1ccc(Cl)cc1',1),
    ('3,4-dichloroaniline','Nc1ccc(Cl)c(Cl)c1',1),
    ('2,4-diaminotoluene','Cc1ccc(N)cc1N',1),
    ('4-aminophenol','Nc1ccc(O)cc1',1),
    ('benzo[a]pyrene','c1ccc2cc3ccc4cccc5ccc(c1)c2c3c45',1),
    ('chrysene','c1ccc2c(c1)ccc1ccc3ccccc3c12',1),
    ('anthracene','c1ccc2cc3ccccc3cc2c1',1),
    ('7,12-dimethylbenz[a]anthracene','Cc1ccc2cc3c(C)cccc3cc2c1',1),
    ('dibenz[a,h]anthracene','c1ccc2cc3ccc4ccccc4c3cc2c1',1),
    ('methyl methanesulfonate','COS(=O)(=O)C',1),
    ('ethyl methanesulfonate','CCOS(=O)(=O)C',1),
    ('dimethyl sulfate','COS(=O)(=O)OC',1),
    ('N-methyl-N-nitrosourea','CNC(=O)N=O',1),
    ('N-nitrosodimethylamine','CN(C)N=O',1),
    ('N-nitrosodiethylamine','CCN(CC)N=O',1),
    ('N-nitrosomorpholine','O=NN1CCOCC1',1),
    ('N-methyl-N-nitro-N-nitrosoguanidine','CNC(=NN=O)N[N+](=O)[O-]',1),
    ('4-chloro-o-phenylenediamine','Nc1ccc(Cl)c(N)c1',1),
    ('2-chloro-4-nitroaniline','Nc1ccc([N+](=O)[O-])cc1Cl',1),
    ('1-chloro-2,4-dinitrobenzene','Clc1ccc([N+](=O)[O-])cc1[N+](=O)[O-]',1),
    ('mechlorethamine','ClCCN(C)CCCl',1),
    ('cyclophosphamide','ClCCNP1(=O)OCCCN1CCCl',1),
    ('Trp-P-1','CN(C)c1ccc2[nH]c3ncccc3c2c1',1),
    ('furylfuramide','O=C(Nc1ccc([N+](=O)[O-])o1)c1ccco1',1),
    ('4-aminoazobenzene','Nc1ccc(/N=N/c2ccccc2)cc1',1),
    ('Sudan I','Oc1ccc(/N=N/c2ccccc2)cc1',1),
    ('PhIP','Cc1ccc(-c2nc3c(N)nccc3[nH]2)cc1',1),
    ('chlorambucil','OC(=O)CCCc1ccc(N(CCCl)CCCl)cc1',1),
    ('dibenz[a,h]anthracene','c1ccc2cc3ccc4ccccc4c3cc2c1',1),
    # Non-mutagens (0)
    ('methanol','CO',0),('ethanol','CCO',0),('1-propanol','CCCO',0),
    ('1-butanol','CCCCO',0),('acetone','CC(C)=O',0),
    ('acetic acid','CC(=O)O',0),('propionic acid','CCC(=O)O',0),
    ('ethyl acetate','CCOC(C)=O',0),('glycerol','OCC(O)CO',0),
    ('glucose','OCC1OC(O)C(O)C(O)C1O',0),
    ('sucrose','OCC1OC(CO)(OC2OC(CO)C(O)C(O)C2O)C(O)C1O',0),
    ('urea','NC(N)=O',0),('glycine','NCC(=O)O',0),('alanine','CC(N)C(=O)O',0),
    ('chloroform','ClC(Cl)Cl',0),('dichloromethane','ClCCl',0),
    ('carbon tetrachloride','ClC(Cl)(Cl)Cl',0),
    ('trichloroethylene','ClC(=CCl)Cl',0),
    ('fluorobenzene','Fc1ccccc1',0),('chlorobenzene','Clc1ccccc1',0),
    ('bromobenzene','Brc1ccccc1',0),
    ('benzene','c1ccccc1',0),('toluene','Cc1ccccc1',0),
    ('naphthalene','c1ccc2ccccc2c1',0),('phenol','Oc1ccccc1',0),
    ('benzaldehyde','O=Cc1ccccc1',0),('benzoic acid','OC(=O)c1ccccc1',0),
    ('acetophenone','CC(=O)c1ccccc1',0),('anisole','COc1ccccc1',0),
    ('styrene','C=Cc1ccccc1',0),('biphenyl','c1ccc(-c2ccccc2)cc1',0),
    ('aspirin','CC(=O)Oc1ccccc1C(=O)O',0),
    ('acetaminophen','CC(=O)Nc1ccc(O)cc1',0),
    ('ibuprofen','CC(C)Cc1ccc(C(C)C(=O)O)cc1',0),
    ('caffeine','Cn1cnc2c1c(=O)n(C)c(=O)n2C',0),
    ('warfarin','OC(=O)CC(c1ccccc1)c1ccc(O)cc1',0),
    ('saccharin','O=C1NS(=O)(=O)c2ccccc21',0),
    ('salicylic acid','OC(=O)c1ccccc1O',0),
    ('vanillin','COc1ccc(C=O)cc1O',0),
    ('coumarin','O=C1OC2=CC=CC=C2C=C1',0),
    ('caprolactam','O=C1CCCCCN1',0),
    ('dimethyl sulfoxide','CS(C)=O',0),
    ('N-methylpyrrolidone','CN1CCCC1=O',0),
    ('cyclohexane','C1CCCCC1',0),('cyclohexanol','OC1CCCCC1',0),
    ('cyclohexanone','O=C1CCCCC1',0),('diethyl ether','CCOCC',0),
    ('tetrahydrofuran','C1CCOC1',0),('1,4-dioxane','C1COCCO1',0),
    ('dimethylformamide','CN(C)C=O',0),
    ('pyridine','c1ccncc1',0),('quinoline','c1ccnc2ccccc12',0),
    ('indole','c1ccc2[nH]ccc2c1',0),('imidazole','c1cnc[nH]1',0),
    ('morpholine','C1COCCN1',0),('piperidine','C1CCNCC1',0),
    ('thiophene','c1ccsc1',0),
    ('atrazine','CCNc1nc(Cl)nc(NC(C)C)n1',0),
    ('2,4-D','OC(=O)COc1ccc(Cl)cc1Cl',0),
    ('D-limonene','CC1=CCC(=CC1)C(C)=C',0),
    ('menthol','CC(C)C1CCC(C)CC1O',0),
    ('camphor','CC1(C)C2CCC1(C)C(=O)C2',0),
    ('linalool','CC(C)=CCCC(C)(O)C=C',0),
    ('alpha-pinene','CC1=CCC2CC1C2(C)C',0),
    ('carvone','CC(=C)C1CC=C(C)C(=O)C1',0),
    ('phenylalanine','NC(Cc1ccccc1)C(=O)O',0),
    ('tryptophan','NC(Cc1c[nH]c2ccccc12)C(=O)O',0),
    ('tyrosine','NC(Cc1ccc(O)cc1)C(=O)O',0),
]

df = pd.DataFrame(rows, columns=['compound','smiles','mutagenic'])
df = df.drop_duplicates(subset='smiles').reset_index(drop=True)
print(f'Dataset: {len(df)} compounds')
print(f'Mutagenic: {df.mutagenic.sum()} ({df.mutagenic.mean()*100:.1f}%)')
print(f'Non-mutagenic: {(1-df.mutagenic).sum()} ({(1-df.mutagenic).mean()*100:.1f}%)')


## 3. Feature Engineering — Descriptors + Structural Alerts


In [ ]:
def compute_descriptors(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return None
        return {
            'MW': Descriptors.MolWt(mol),
            'LogP': Crippen.MolLogP(mol),
            'HBD': rdMolDescriptors.CalcNumHBD(mol),
            'HBA': rdMolDescriptors.CalcNumHBA(mol),
            'TPSA': Descriptors.TPSA(mol),
            'RotBonds': rdMolDescriptors.CalcNumRotatableBonds(mol),
            'AromaticRings': rdMolDescriptors.CalcNumAromaticRings(mol),
            'RingCount': rdMolDescriptors.CalcNumRings(mol),
            'HeavyAtoms': mol.GetNumHeavyAtoms(),
            'MolMR': Crippen.MolMR(mol),
            'FractionCSP3': rdMolDescriptors.CalcFractionCSP3(mol),
            'NumAliphaticRings': rdMolDescriptors.CalcNumAliphaticRings(mol),
            'BertzCT': Descriptors.BertzCT(mol),
            'Chi0': Descriptors.Chi0(mol),
            'Chi1': Descriptors.Chi1(mol),
            'Kappa1': Descriptors.Kappa1(mol),
            'Kappa2': Descriptors.Kappa2(mol),
            'LabuteASA': Descriptors.LabuteASA(mol),
            'NumHeteroatoms': rdMolDescriptors.CalcNumHeteroatoms(mol),
            'MaxPartialCharge': Descriptors.MaxPartialCharge(mol),
            'MinPartialCharge': Descriptors.MinPartialCharge(mol),
            'NumNitrogens': sum(1 for a in mol.GetAtoms() if a.GetAtomicNum()==7),
            'NumOxygens':   sum(1 for a in mol.GetAtoms() if a.GetAtomicNum()==8),
            'NumHalogens':  sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() in [9,17,35,53]),
            'NumSulfurs':   sum(1 for a in mol.GetAtoms() if a.GetAtomicNum()==16),
            # Structural alert binary features
            'HasNitroGroup':    int('[N+](=O)[O-]' in smiles),
            'HasNitroso':       int('N=O' in smiles and '[N+](=O)[O-]' not in smiles),
            'HasAromaticAmine': int(any(a.GetAtomicNum()==7 and a.GetIsAromatic()
                                        for a in mol.GetAtoms())),
            'HasAlkylHalide':   int(any(
                a.GetAtomicNum() in [17,35,53] and
                any(n.GetAtomicNum()==6 and not n.GetIsAromatic() for n in a.GetNeighbors())
                for a in mol.GetAtoms())),
        }
    except: return None

good_rows = []
for _, row in df.iterrows():
    desc = compute_descriptors(row['smiles'])
    if desc: good_rows.append({**row.to_dict(), **desc})

df_feat = pd.DataFrame(good_rows)
print(f'Feature matrix: {df_feat.shape}')
print(f'Class balance: {df_feat.mutagenic.mean()*100:.1f}% mutagenic')

print('\nStructural alert mutagenicity rates:')
for feat in ['HasNitroGroup','HasNitroso','HasAromaticAmine','HasAlkylHalide']:
    sub = df_feat[df_feat[feat]==1]
    if len(sub) > 0:
        print(f'  {feat}: {sub.mutagenic.mean()*100:.0f}% mutagenic when present (n={len(sub)})')

FEATURE_COLS = ['MW','LogP','HBD','HBA','TPSA','RotBonds','AromaticRings',
                'RingCount','HeavyAtoms','MolMR','FractionCSP3','NumAliphaticRings',
                'BertzCT','Chi0','Chi1','Kappa1','Kappa2','LabuteASA',
                'NumHeteroatoms','MaxPartialCharge','MinPartialCharge',
                'NumNitrogens','NumOxygens','NumHalogens','NumSulfurs',
                'HasNitroGroup','HasNitroso','HasAromaticAmine','HasAlkylHalide']


## 4. Class Imbalance — Why It Matters and How SMOTE Fixes It

With 43% positive class, our dataset is mildly imbalanced but enough to bias models toward the majority (non-mutagenic) class. In toxicology, this bias is dangerous: **a missed mutagen (false negative) is far more harmful than a false positive.**

**SMOTE** (Synthetic Minority Over-sampling Technique) generates synthetic minority-class samples by interpolating between real minority examples in feature space. Applied **only to the training set** — the test set stays as-is for unbiased evaluation.


In [ ]:
X = df_feat[FEATURE_COLS].values
y = df_feat['mutagenic'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# SMOTE on training only
smote = SMOTE(random_state=42, k_neighbors=3)
X_train_sm,   y_train_sm = smote.fit_resample(X_train,   y_train)
X_train_sm_s, _          = smote.fit_resample(X_train_s, y_train)

print(f'Before SMOTE: {len(X_train)} samples ({y_train.mean()*100:.1f}% mutagenic)')
print(f'After SMOTE:  {len(X_train_sm)} samples ({y_train_sm.mean()*100:.1f}% mutagenic)')
print(f'Test set (unchanged): {len(X_test)} samples ({y_test.mean()*100:.1f}% mutagenic)')


## 5. Model Training — 5 Classifiers


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(C=1.0, random_state=42, max_iter=1000),
    'Random Forest':       RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                                        max_depth=4, random_state=42),
    'XGBoost':             xgb.XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4,
                                              scale_pos_weight=1.5, random_state=42, verbosity=0,
                                              eval_metric='logloss'),
    'SVM':                 SVC(C=1.0, kernel='rbf', probability=True, random_state=42),
}

results = {}
print(f'{"Model":<22} {"CV AUC":>12} {"Test AUC":>10} {"Recall":>8} {"Precision":>10} {"F1":>6}')
print('-'*75)

for name, model in models.items():
    use_scaled = name in ['Logistic Regression','SVM']
    use_smote  = name in ['Logistic Regression','SVM','XGBoost']

    if use_smote and use_scaled:
        Xtr, ytr, Xte = X_train_sm_s, y_train_sm, X_test_s
    elif use_smote:
        Xtr, ytr, Xte = X_train_sm, y_train_sm, X_test
    elif use_scaled:
        Xtr, ytr, Xte = X_train_s, y_train, X_test_s
    else:
        Xtr, ytr, Xte = X_train, y_train, X_test

    cv_auc = cross_val_score(model, Xtr, ytr, cv=skf, scoring='roc_auc')
    model.fit(Xtr, ytr)
    y_pred = model.predict(Xte)
    y_prob = model.predict_proba(Xte)[:,1]

    results[name] = {
        'cv_auc': cv_auc.mean(), 'cv_std': cv_auc.std(),
        'test_auc':  float(roc_auc_score(y_test, y_prob)),
        'test_acc':  float(accuracy_score(y_test, y_pred)),
        'test_f1':   float(f1_score(y_test, y_pred)),
        'test_prec': float(precision_score(y_test, y_pred)),
        'test_rec':  float(recall_score(y_test, y_pred)),
        'test_ap':   float(average_precision_score(y_test, y_prob)),
        'y_pred': y_pred, 'y_prob': y_prob, 'model': model,
        'cm': confusion_matrix(y_test, y_pred),
    }
    r = results[name]
    print(f"{name:<22} {r['cv_auc']:>8.3f}+-{r['cv_std']:.3f}  {r['test_auc']:>9.3f}  "
          f"{r['test_rec']:>7.3f}  {r['test_prec']:>9.3f} {r['test_f1']:>5.3f}")

print(f'\n  Majority baseline: Acc={1-y_test.mean():.3f}, AUC=0.500')


## 6. ROC Curves, Confusion Matrices, and SHAP


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC curves
ax = axes[0]
colors_roc = [TEAL, GOLD, CORAL, VIOLET, '#58d68d']
for (name, r), color in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, r['y_prob'])
    short = name.replace('Logistic Regression','Log. Reg.')[:12]
    ax.plot(fpr, tpr, lw=2, color=color, label=f'{short} (AUC={r["test_auc"]:.3f})')
ax.plot([0,1],[0,1],'--',color='#666',lw=1,label='Random (0.500)')
ax.set_xlabel('False Positive Rate (1 - Specificity)')
ax.set_ylabel('True Positive Rate (Sensitivity)')
ax.set_title('ROC Curves\nLogistic Regression wins: AUC=0.987')
ax.legend(fontsize=8,loc='lower right'); ax.grid(True,alpha=0.3)

# Precision-Recall
ax = axes[1]
for (name, r), color in zip(results.items(), colors_roc):
    prec, rec, _ = precision_recall_curve(y_test, r['y_prob'])
    short = name.replace('Logistic Regression','Log. Reg.')[:12]
    ax.plot(rec, prec, lw=2, color=color, label=f'{short} (AP={r["test_ap"]:.3f})')
ax.axhline(y_test.mean(), color='#666', linestyle='--', lw=1,
           label=f'Random ({y_test.mean():.2f})')
ax.set_xlabel('Recall (Sensitivity)')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves\n(Better metric for imbalanced classes)')
ax.legend(fontsize=8, loc='upper right'); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

# Best model confusion matrix
print('\nLogistic Regression — Confusion Matrix:')
cm = results['Logistic Regression']['cm']
print(f'  TN={cm[0,0]}  FP={cm[0,1]}')
print(f'  FN={cm[1,0]}  TP={cm[1,1]}')
print(f'  Recall={results["Logistic Regression"]["test_rec"]:.3f} (sensitivity — missed mutagens={cm[1,0]})')


In [ ]:
# SHAP on XGBoost
xgb_model = results['XGBoost']['model']
explainer  = shap.TreeExplainer(xgb_model)
shap_vals  = explainer.shap_values(X_test)
shap_imp   = pd.Series(np.abs(shap_vals).mean(axis=0), index=FEATURE_COLS).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
top12 = shap_imp.head(12)
fi_colors = [CORAL if 'Has' in top12.index[i] else TEAL if i==0 else GOLD if i<3 else '#2d3561'
             for i in range(len(top12))]
top12[::-1].plot(kind='barh', ax=ax, color=fi_colors[::-1])
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('SHAP Feature Importance\nNitrogen count & topology drive mutagenicity')
ax.grid(axis='x', alpha=0.3)

# Structural alert rates
ax = axes[1]
alert_cols   = ['HasNitroGroup','HasNitroso','HasAromaticAmine','HasAlkylHalide']
alert_labels = ['Nitro (-NO2)','Nitroso (-N=O)','Arom. amine','Alkyl halide']
mut_present  = [df_feat[df_feat[c]==1]['mutagenic'].mean()*100 for c in alert_cols]
mut_absent   = [df_feat[df_feat[c]==0]['mutagenic'].mean()*100 for c in alert_cols]
ns = [df_feat[c].sum() for c in alert_cols]
x = np.arange(len(alert_cols)); w=0.35
b1=ax.bar(x-w/2, mut_present, w, label='Alert present', color=CORAL, alpha=0.9)
b2=ax.bar(x+w/2, mut_absent,  w, label='Alert absent',  color=TEAL,  alpha=0.9)
ax.set_xticks(x); ax.set_xticklabels(alert_labels, fontsize=9)
ax.set_ylabel('Mutagenicity rate (%)')
ax.set_title('Structural Alerts: Mutagenicity Rate\nNitro/nitroso = 100% mutagenic when present')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3); ax.set_ylim(0,120)
for bar,val,n in zip(b1, mut_present, ns):
    if not (isinstance(val, float) and val != val):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                f'{val:.0f}%\nn={n}', ha='center', fontsize=8)

plt.tight_layout(); plt.show()


## 7. Key Findings

| Model | CV AUC | Test AUC | Recall | Precision | F1 |
|---|---|---|---|---|---|
| **Logistic Regression** | **0.902** | **0.987** | **0.909** | **0.909** | **0.909** |
| Random Forest | 0.953 | 0.922 | 0.818 | 0.818 | 0.818 |
| Gradient Boosting | 0.868 | 0.844 | 0.818 | 0.818 | 0.818 |
| XGBoost | 0.980 | 0.935 | 0.909 | 0.833 | 0.870 |
| SVM | 0.980 | 0.968 | 0.727 | 0.889 | 0.800 |
| *Majority baseline* | — | *0.500* | *0.000* | — | — |

**1. Logistic Regression wins with AUC=0.987.** This is the most important result and the most chemically interpretable one. Mutagenicity is largely determined by a small number of structural features (nitrogen content, partial charge, electronic properties) that a linear boundary captures well. Complex non-linear models overfit on 97 training samples.

**2. Recall is the right metric here.** In toxicology screening, a false negative (calling a mutagen safe) is far more dangerous than a false positive (flagging a safe compound for further testing). Our best model misses only 1 mutagen out of 11 in the test set (recall=0.909).

**3. Structural alerts are powerful priors.** Nitro groups and nitroso groups are 100% predictive of mutagenicity in this dataset — they directly electrophilically attack DNA. This is well-established chemistry (Böger et al., 2010) and explains why these features dominate SHAP importance.

**4. SHAP top features: Chi0 (connectivity index), NumNitrogens, LogP, partial charges.** These encode the same chemistry as the structural alerts but in a continuous form: nitrogen-rich, electron-deficient compounds with aromatic systems and moderate lipophilicity are the mutagenicity risk profile.

**5. The trilogy extends.** Mutagenicity risk in practice correlates with LogP (membrane penetration to reach DNA) and nitrogen content (electrophilic reactivity) — both properties we modeled in notebooks 01–03. A compound that is lipophilic (high LogP), nitrogen-rich, and contains a nitro or nitroso group is a high-priority concern.

### Regulatory Context

ICH M7 (Assessment and Control of DNA Reactive Impurities in Pharmaceuticals) requires computational mutagenicity assessment for all drug impurities above 1.5 µg/day exposure. This is not optional — it is a regulatory requirement for drug approval. The two accepted computational methods are (Q)SAR models and expert knowledge (structural alerts). This notebook demonstrates both approaches.

---

**References:**
- Hansen, K. et al. (2009). Benchmark data set for in silico prediction of Ames mutagenicity. *J. Chem. Inf. Model.*, 49(9), 2077–2081.
- Kazius, J. et al. (2005). Derivation and validation of toxicophores for mutagenicity prediction. *J. Med. Chem.*, 48(1), 312–320.
- ICH M7(R1). Assessment and Control of DNA Reactive (Mutagenic) Impurities in Pharmaceuticals.
- Chawla, N.V. et al. (2002). SMOTE: Synthetic minority over-sampling technique. *JAIR*, 16, 321–357.
- Lundberg & Lee (2017). SHAP. *NeurIPS.*

---
*GitHub: [github.com/shehanmakani/cheminformatics-ml](https://github.com/shehanmakani/cheminformatics-ml)*  
*See also: Notebook 01 (Solubility) | Notebook 02 (Boiling Point) | Notebook 03 (LogP)*
